In [1]:
import os
import joblib
import numpy as np
import pandas as pd
from datetime import datetime
from typing import List, Dict, Any, Tuple

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, log_loss
from xgboost import XGBClassifier

In [2]:
# pipeline.py


# ---------- PARAMETERS ----------
# df_model must already contain:
# - one row per prediction opportunity
# - a column "SaleTransactionDate" of dtype datetime
# - target column "next_target_group" (string label of category/family)
# - any engineered feature columns (pref_*, cum_spent, days_since_last_purchase, ...)

df_model = pd.read_csv("../data/transformed/df_model_final_family_2.csv")
TARGET_COL = "next_target_group"         # string label (not encoded)
DATE_COL = "SaleTransactionDate"
ID_COL = "ClientID"

MODEL_DIR = "./models"
os.makedirs(MODEL_DIR, exist_ok=True)

# Time split: use a date cutoff for test (adjust to your data)
TEST_CUTOFF = pd.Timestamp("2025-01-01", tz='UTC')   # example: all rows >= this => test
VALIDATION_WINDOW_DAYS = 90                # last 90 days before TEST_CUTOFF used for validation

RANDOM_STATE = 42
TOP_K = 5

In [3]:
df_model["SaleTransactionDate"].max()

'2025-02-14 00:00:00+00:00'

In [4]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df_model["y"] = le.fit_transform(df_model["next_target_group"])

In [5]:
le.inverse_transform(df_model["y"]) # will be used to compare predictions in the end


array(['Select Ultimate', 'Nike Away Jersey', 'Wilson US Open', ...,
       'Wilson US Open', 'Wilson US Open', 'Victor Gold Champion'],
      shape=(867125,), dtype=object)

In [6]:
## For reference, what does "y" corresponds to :
df_model["next_target_group"]

0                         Select Ultimate
1                        Nike Away Jersey
2                          Wilson US Open
3                          Wilson US Open
4                             Puma Future
                       ...               
867120                  Penn Championship
867121    Spalding NBA Official Game Ball
867122                     Wilson US Open
867123                     Wilson US Open
867124               Victor Gold Champion
Name: next_target_group, Length: 867125, dtype: object

In [7]:
exclude = [ID_COL, DATE_COL, TARGET_COL, "y", "Unnamed: 0.1", "Unnamed: 0"]
candidate = [c for c in df_model.columns if c not in exclude]
numeric_cols = df_model[candidate].select_dtypes(include=["number"]).columns.tolist()
categorical_cols = [c for c in candidate if c not in numeric_cols]

In [8]:
numeric_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])
categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("ord", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1))
])
preprocessor = ColumnTransformer(
    [
        ("num", numeric_pipe, numeric_cols),
        ("cat", categorical_pipe, categorical_cols),
    ],
    remainder="drop",
    sparse_threshold=0
)

In [9]:
date_col = DATE_COL
test_cutoff = TEST_CUTOFF
val_window_days = VALIDATION_WINDOW_DAYS


df = df_model.copy()
df[DATE_COL] = pd.to_datetime(df[DATE_COL])
test_mask = df[date_col] >= test_cutoff
train_mask = df[date_col] < test_cutoff

train_df = df[train_mask].copy()
test_df = df[test_mask].copy()

In [10]:
X_train = preprocessor.fit_transform(train_df[numeric_cols + categorical_cols])
y_train = train_df["y"].values

X_test = preprocessor.transform(test_df[numeric_cols + categorical_cols]) if len(test_df) else None
y_test = test_df["y"].values if len(test_df) else None

In [11]:
model = XGBClassifier(
    objective="multi:softprob",
    eval_metric="mlogloss",
    n_estimators=300,        # 1000 -> 300
    learning_rate=0.1,       # 0.05 -> 0.1 (need fewer trees)
    max_depth=0,
    max_leaves=32,           # 64 -> 32
    grow_policy="lossguide",
    min_child_weight=10,     # 5 -> 10
    subsample=0.7,           # 0.8 -> 0.7
    colsample_bytree=0.7,    # 0.8 -> 0.7
    gamma=0.2,
    tree_method="hist",
    max_bin=128,             # 256 -> 128
    n_jobs=-1,
    random_state=RANDOM_STATE
)

model.fit(X_train, y_train)


In [ ]:
def top_k_accuracy(y_true: np.ndarray, y_proba: np.ndarray, k: int) -> float:
    topk = np.argsort(y_proba, axis=1)[:, -k:][:, ::-1]  # top k indices
    hits = 0
    for i, true in enumerate(y_true):
        if true in topk[i]:
            hits += 1
    return hits / len(y_true)

def mrr_score(y_true: np.ndarray, y_proba: np.ndarray) -> float:
    order = np.argsort(y_proba, axis=1)[:, ::-1]  # descending order
    rr_sum = 0.0
    n = len(y_true)
    for i, true in enumerate(y_true):
        ranks = np.where(order[i] == true)[0]
        if ranks.size > 0:
            rr_sum += 1.0 / (ranks[0] + 1.0)
    return rr_sum / n

In [ ]:
def eval_and_print(X, y, split_name="set"):
        if X is None or len(y)==0:
            print(f"No data for {split_name}")
            return
        proba = model.predict_proba(X)
        acc = accuracy_score(y, np.argmax(proba, axis=1))
        loss = log_loss(y, proba)
        top1 = top_k_accuracy(y, proba, 1)
        top5 = top_k_accuracy(y, proba, TOP_K)
        mrr = mrr_score(y, proba)
        print(f"== {split_name} metrics ==")
        print(f"Accuracy: {acc:.4f}; LogLoss: {loss:.4f}; Top-1: {top1:.4f}; Top-{TOP_K}: {top5:.4f}; MRR: {mrr:.4f}")

eval_and_print(X_train, y_train, "train")
eval_and_print(X_test, y_test, "test")

    # Feature importances
try:
    importances = model.get_booster().get_score(importance_type="gain")
    sorted_imp = sorted(importances.items(), key=lambda x: x[1], reverse=True)[:30]
    print("Top feature importances (gain):")
    for feat, val in sorted_imp:
        print(feat, val)
except Exception as e:
    print("Could not extract feature importances:", e)

In [ ]:
def predict_top_n_for_client(
    df_row: pd.DataFrame,                 # one row of features for the client (pandas DataFrame with same columns as training features)
    preprocessor,
    model: XGBClassifier,
    label_encoder: LabelEncoder,
    top_n: int = 5
) -> List[Tuple[str, float]]:
    """
    Returns list of (label, prob) sorted by prob desc
    df_row should contain the feature columns (not target nor IDs), shape (1, n_features)
    """
    X_proc = preprocessor.transform(df_row)
    proba = model.predict_proba(X_proc)[0]   # shape (n_classes,)
    top_idx = np.argsort(proba)[-top_n:][::-1]
    labels = label_encoder.inverse_transform(top_idx)
    return list(zip(labels, proba[top_idx]))

In [ ]:
def save_artifacts(preprocessor, label_encoder, model, out_dir=MODEL_DIR):
    joblib.dump(preprocessor, os.path.join(out_dir, "preprocessor.joblib"))
    joblib.dump(label_encoder, os.path.join(out_dir, "label_encoder.joblib"))
    joblib.dump(model, os.path.join(out_dir, "xgb_model.joblib"))

def load_artifacts(out_dir=MODEL_DIR):
    preprocessor = joblib.load(os.path.join(out_dir, "preprocessor.joblib"))
    label_encoder = joblib.load(os.path.join(out_dir, "label_encoder.joblib"))
    model = joblib.load(os.path.join(out_dir, "xgb_model.joblib"))
    return preprocessor, label_encoder, model